# 02 -- Exploratory data analysis

**What this notebook does (plain English):** A few clear pictures of *what makes
a mortgage risky*. We look at how the default rate changes with the borrower's
**credit score**, with the **loan-to-value** ratio (how big the loan is versus
the home's value), and across the three **vintages**. Charts are saved for the
README.

**Headline result:** default rate falls steadily as credit score rises and rises
sharply as loan-to-value climbs -- the two classic mortgage risk drivers.

In [1]:
import sys, os
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'src')):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')
print('project root:', ROOT)

project root: D:\Jane\Job Search\Github\bank\github project\freddie mac mortgage


In [2]:
# Load the base table and set up plotting (headless backend for saving files).
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.output import save_csv
base = pd.read_parquet('data/processed/analysis_base.parquet')
os.makedirs('outputs/charts', exist_ok=True)

In [3]:
# Default rate by credit-score band.
by_score = base.groupby('credit_score_band', observed=True)['ever_default'].mean()
ax = by_score.plot(kind='bar', color='#4C72B0', title='Default rate by credit-score band')
ax.set_ylabel('default rate'); plt.tight_layout()
plt.savefig('outputs/charts/default_by_credit_score.png', dpi=110); plt.close()

In [4]:
# Default rate by loan-to-value band.
by_ltv = base.groupby('ltv_band', observed=True)['ever_default'].mean()
ax = by_ltv.plot(kind='bar', color='#C44E52', title='Default rate by loan-to-value band')
ax.set_ylabel('default rate'); plt.tight_layout()
plt.savefig('outputs/charts/default_by_ltv.png', dpi=110); plt.close()

In [5]:
# One-page risk-by-driver summary table (the saved result for this notebook).
rows = []
for band, v in base.groupby('credit_score_band', observed=True)['ever_default'].mean().items():
    rows.append({'driver': 'credit_score', 'band': band, 'default_rate': round(v, 4)})
for band, v in base.groupby('ltv_band', observed=True)['ever_default'].mean().items():
    rows.append({'driver': 'ltv', 'band': band, 'default_rate': round(v, 4)})
for band, v in base.groupby('vintage_year')['ever_default'].mean().items():
    rows.append({'driver': 'vintage', 'band': str(band), 'default_rate': round(v, 4)})
risk_by_driver = pd.DataFrame(rows)
save_csv(risk_by_driver, 'outputs/tables/02_risk_by_driver.csv')
risk_by_driver

,driver,band,default_rate
0,credit_score,<620,0.2901
1,credit_score,620-659,0.2181
2,credit_score,660-699,0.1438
3,credit_score,700-739,0.0873
4,credit_score,740-779,0.0452
5,credit_score,780+,0.0201
6,ltv,<60,0.0314
7,ltv,60-69,0.0621
8,ltv,70-79,0.0858
9,ltv,80-89,0.0882


**Reading the table:** weaker credit scores and higher loan-to-value both
line up with higher default rates, and every band is worse in the crisis years.
These are the drivers we feed into the PD model next.